# 🚆 ETL — Données ferroviaires européennes

**Pipeline complet : Extract → Transform → Load**

| Fichier | Contenu |
|---|---|
| `co2_comparaison_europe.csv` | Émissions CO₂ par mode de transport par pays |
| `france_rail_OD_flows.csv` | Flux OD ferroviaires France |
| `germany_rail_OD_flows.csv` | Flux OD ferroviaires Allemagne |
| `italy_rail_OD_flows.csv` | Flux OD ferroviaires Italie |
| `spain_rail_OD_flows.csv` | Flux OD ferroviaires Espagne |
| `switzerland_rail_OD_flows.csv` | Flux OD ferroviaires Suisse |
| `cross_border_rail_OD_IT_DE_CH_ES.csv` | Flux transfrontaliers (IT, DE, CH, ES) |

---
## 0. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Chemins (adapter si besoin) ──────────────────────────────────
DATA_DIR   = "."        # dossier contenant les CSV sources
OUTPUT_DIR = "./output" # dossier de sortie
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
print('✅ Prêt')

---
## 1. EXTRACT — Chargement des fichiers bruts

In [ ]:
def load_csv(filename, sep=';'):
    """Charge un CSV et affiche un résumé rapide."""
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path, sep=sep, encoding='utf-8-sig')
    print(f'📂  {filename:<50}  {df.shape[0]:>5} lignes | {df.shape[1]} colonnes')
    return df

print('Chargement des fichiers...\n')
df_co2        = load_csv('co2_comparaison_europe.csv')
df_france     = load_csv('france_rail_OD_flows.csv')
df_germany    = load_csv('germany_rail_OD_flows.csv')
df_italy      = load_csv('italy_rail_OD_flows.csv')
df_spain      = load_csv('spain_rail_OD_flows.csv')
df_switz      = load_csv('switzerland_rail_OD_flows.csv')
df_crossbord  = load_csv('cross_border_rail_OD_IT_DE_CH_ES.csv')
print('\n✅ Tous les fichiers chargés.')

---
## 2. EXPLORE — Aperçu des données brutes

In [ ]:
print('=== CO2 ===')
display(df_co2)

print('\n=== Flux France (5 premières lignes) ===')
display(df_france.head())

print('\n=== Flux Allemagne (5 premières lignes) ===')
display(df_germany.head())

print('\n=== Flux cross-border (5 premières lignes) ===')
display(df_crossbord.head())

In [ ]:
# Résumé qualité des données brutes
datasets = {
    'co2'          : df_co2,
    'france'       : df_france,
    'germany'      : df_germany,
    'italy'        : df_italy,
    'spain'        : df_spain,
    'switzerland'  : df_switz,
    'cross_border' : df_crossbord,
}

print(f"{'Dataset':<16} {'Lignes':>6} {'Colonnes':>9} {'Manquants':>10}")
print('-' * 46)
for name, df in datasets.items():
    missing = df.isnull().sum().sum()
    print(f"{name:<16} {df.shape[0]:>6} {df.shape[1]:>9} {missing:>10}")

---
## 3. TRANSFORM — Nettoyage & standardisation

### 3.1 Table CO₂

In [ ]:
def transform_co2(df):
    df = df.copy()
    df.columns = [
        'country', 'emissions_train_g_km', 'emissions_car_g_km',
        'emissions_plane_g_km', 'co2_saved_vs_car_g_km',
        'co2_saved_vs_plane_g_km', 'reduction_vs_car_pct'
    ]
    df['country'] = df['country'].str.strip()
    num_cols = df.columns[1:]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    print(f'✅ CO2 transformé : {df.shape}')
    return df

df_co2_clean = transform_co2(df_co2)
display(df_co2_clean)

### 3.2 Flux OD nationaux (France, Allemagne, Italie, Espagne, Suisse)

In [ ]:
# Mapping pays → (code ISO, label anglais, label CO2)
COUNTRY_MAP = {
    'france'      : ('FR', 'France',      'France'),
    'germany'     : ('DE', 'Germany',     'Allemagne'),
    'italy'       : ('IT', 'Italy',       'Italie'),
    'spain'       : ('ES', 'Spain',       'Espagne'),
    'switzerland' : ('CH', 'Switzerland', 'Suisse'),
}

# Colonnes standardisées communes aux 5 fichiers nationaux
NATIONAL_COLS = [
    'year', 'origin_station', 'destination_station',
    'origin_city', 'destination_city',
    'origin_region', 'destination_region',
    'passengers_millions', 'service_type', 'source'
]

def transform_od_national(df, country_key):
    """Nettoyage générique pour les flux OD nationaux."""
    df = df.copy()
    iso, label, _ = COUNTRY_MAP[country_key]

    df.columns = NATIONAL_COLS

    # Métadonnées pays
    df['country']     = label
    df['country_iso'] = iso
    df['flow_type']   = 'domestic'

    # Cast types
    df['year']               = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['passengers_millions'] = pd.to_numeric(df['passengers_millions'], errors='coerce')

    # Nettoyage textes
    text_cols = ['origin_station', 'destination_station',
                 'origin_city', 'destination_city',
                 'origin_region', 'destination_region']
    for col in text_cols:
        df[col] = df[col].str.strip()

    # Suppression doublons
    before = len(df)
    df.drop_duplicates(subset=['year', 'origin_station', 'destination_station'], inplace=True)
    removed = before - len(df)
    if removed:
        print(f'  ⚠️  {removed} doublon(s) supprimé(s) dans {label}')

    # Suppression lignes sans passagers
    df.dropna(subset=['passengers_millions'], inplace=True)

    print(f'✅ {label:<15} transformé : {df.shape[0]} lignes')
    return df

df_france_clean  = transform_od_national(df_france,  'france')
df_germany_clean = transform_od_national(df_germany, 'germany')
df_italy_clean   = transform_od_national(df_italy,   'italy')
df_spain_clean   = transform_od_national(df_spain,   'spain')
df_switz_clean   = transform_od_national(df_switz,   'switzerland')

### 3.3 Flux cross-border

In [ ]:
def transform_crossborder(df):
    df = df.copy()
    df.columns = [
        'year',
        'origin_country_iso', 'origin_station', 'origin_city', 'origin_region',
        'destination_country_iso', 'destination_station', 'destination_city', 'destination_region',
        'passengers_millions', 'service_type', 'source'
    ]
    df['flow_type'] = 'cross_border'

    df['year']                = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df['passengers_millions'] = pd.to_numeric(df['passengers_millions'], errors='coerce')

    text_cols = ['origin_station', 'destination_station',
                 'origin_city', 'destination_city',
                 'origin_region', 'destination_region']
    for col in text_cols:
        df[col] = df[col].str.strip()

    # Normaliser les noms de pays ISO (ex: 'Italy' → 'IT')
    country_to_iso = {
        'France': 'FR', 'Germany': 'DE', 'Italy': 'IT',
        'Spain': 'ES', 'Switzerland': 'CH'
    }
    df['origin_country_iso']      = df['origin_country_iso'].map(country_to_iso).fillna(df['origin_country_iso'])
    df['destination_country_iso'] = df['destination_country_iso'].map(country_to_iso).fillna(df['destination_country_iso'])

    df.dropna(subset=['passengers_millions'], inplace=True)
    print(f'✅ Cross-border transformé : {df.shape[0]} lignes')
    return df

df_crossbord_clean = transform_crossborder(df_crossbord)
display(df_crossbord_clean.head())

---
## 4. CONSOLIDATION — Table OD unifiée

In [ ]:
# Schéma commun minimal pour le concat
COMMON_COLS = [
    'year', 'origin_station', 'destination_station',
    'origin_city', 'destination_city',
    'origin_region', 'destination_region',
    'passengers_millions', 'service_type', 'flow_type', 'source'
]

# Flux nationaux : on ajoute origin/destination_country_iso identiques
national_frames = []
for df, iso in [
    (df_france_clean,  'FR'),
    (df_germany_clean, 'DE'),
    (df_italy_clean,   'IT'),
    (df_spain_clean,   'ES'),
    (df_switz_clean,   'CH'),
]:
    tmp = df[COMMON_COLS].copy()
    tmp['origin_country_iso']      = iso
    tmp['destination_country_iso'] = iso
    national_frames.append(tmp)

# Flux cross-border
cb = df_crossbord_clean[COMMON_COLS + ['origin_country_iso', 'destination_country_iso']].copy()

# Concatenation
df_all_od = pd.concat(national_frames + [cb], ignore_index=True)

print(f'✅ Table OD consolidée : {df_all_od.shape[0]:,} lignes | {df_all_od.shape[1]} colonnes')
display(df_all_od.sample(5, random_state=42))

### 4.1 Enrichissement — jointure avec CO₂

In [ ]:
# Mapping ISO → nom dans la table CO2 (noms en français)
ISO_TO_CO2_COUNTRY = {
    'FR': 'France',
    'DE': 'Allemagne',
    'IT': 'Italie',
    'ES': 'Espagne',
    'CH': 'Suisse',
}

df_all_od['co2_country_key'] = df_all_od['origin_country_iso'].map(ISO_TO_CO2_COUNTRY)

df_enriched = df_all_od.merge(
    df_co2_clean[['country', 'emissions_train_g_km', 'emissions_car_g_km',
                  'co2_saved_vs_car_g_km', 'reduction_vs_car_pct']],
    left_on='co2_country_key',
    right_on='country',
    how='left'
).drop(columns=['country', 'co2_country_key'])

print(f'✅ Table enrichie : {df_enriched.shape}')
print(f"   Taux de match CO2 : {df_enriched['emissions_train_g_km'].notna().mean():.1%}")
display(df_enriched.head())

---
## 5. CONTRÔLES QUALITÉ & STATISTIQUES

In [ ]:
print('=== Répartition par pays et type de flux ===')
display(
    df_enriched.groupby(['origin_country_iso', 'flow_type'])
               .agg(
                   nb_routes          = ('passengers_millions', 'count'),
                   total_passengers_M = ('passengers_millions', 'sum'),
               )
               .round(2)
               .reset_index()
               .sort_values('total_passengers_M', ascending=False)
)

In [ ]:
print('=== Top 10 des routes les plus fréquentées ===')
top10 = (
    df_enriched
    .groupby(['origin_country_iso', 'origin_city', 'destination_city'])
    ['passengers_millions'].sum()
    .reset_index()
    .sort_values('passengers_millions', ascending=False)
    .head(10)
)
display(top10)

In [ ]:
print('=== Évolution totale par année ===')
display(
    df_enriched.groupby('year')
               .agg(
                   nb_routes          = ('passengers_millions', 'count'),
                   total_passengers_M = ('passengers_millions', 'sum'),
               )
               .round(2)
)

In [ ]:
print('=== Comparatif CO₂ par pays (train vs voiture) ===')
display(df_co2_clean[['country', 'emissions_train_g_km', 'emissions_car_g_km',
                       'reduction_vs_car_pct']].sort_values('emissions_train_g_km'))

---
## 6. LOAD — Export des tables finales

In [ ]:
# ── Table 1 : flux OD enrichis (table principale) ────────────────
out_od = os.path.join(OUTPUT_DIR, 'od_flows_enriched.csv')
df_enriched.to_csv(out_od, index=False, sep=';', encoding='utf-8-sig')
print(f'💾 od_flows_enriched.csv     → {len(df_enriched):,} lignes')

# ── Table 2 : CO2 nettoyé ─────────────────────────────────────────
out_co2 = os.path.join(OUTPUT_DIR, 'co2_clean.csv')
df_co2_clean.to_csv(out_co2, index=False, sep=';', encoding='utf-8-sig')
print(f'💾 co2_clean.csv             → {len(df_co2_clean):,} lignes')

# ── Table 3 : agrégat par pays/année/type ─────────────────────────
df_agg = (
    df_enriched
    .groupby(['year', 'origin_country_iso', 'flow_type'])
    .agg(
        nb_routes          = ('passengers_millions', 'count'),
        total_passengers_M = ('passengers_millions', 'sum'),
        avg_passengers_M   = ('passengers_millions', 'mean'),
        max_route_M        = ('passengers_millions', 'max'),
    )
    .round(3)
    .reset_index()
)
out_agg = os.path.join(OUTPUT_DIR, 'od_aggregated.csv')
df_agg.to_csv(out_agg, index=False, sep=';', encoding='utf-8-sig')
print(f'💾 od_aggregated.csv         → {len(df_agg):,} lignes')

print(f'\n✅ Export terminé — dossier : {os.path.abspath(OUTPUT_DIR)}')

---
## 7. Récapitulatif du pipeline

| Étape | Description | Résultat |
|---|---|---|
| **Extract** | Lecture de 7 CSV (séparateur `;`, encoding `utf-8-sig`) | ~2 500 lignes brutes |
| **Transform** | Renommage colonnes, cast types, suppression doublons/NaN, normalisation ISO | Schéma unifié |
| **Consolidation** | Concat des 5 pays + cross-border + jointure CO₂ | 1 table enrichie |
| **Load** | Export en 3 CSV structurés dans `./output/` | Prêt pour analyse / BI |